# MRT Distance Pipeline

This notebook geocodes every unique HDB resale flat address and finds its nearest MRT station (excluding LRT) using the [OneMap Singapore API](https://www.onemap.gov.sg/apidocs/).

## Inputs
- `../merged_data/merged_hdb_resale_with_macro.csv` — the enriched HDB resale dataset
- `../.env` — OneMap credentials (fill in before running):
  ```
  ONEMAP_EMAIL=your_email
  ONEMAP_PASSWORD=your_password
  ```

## How to Run
1. Fill in your OneMap credentials in `../.env`
2. Run cells top to bottom — **Phase 3 and Phase 4 are resume-safe**: if the notebook is interrupted, re-running will skip already-cached addresses.

## Output Files
| File | Description |
|------|-------------|
| `../merged_data/hdb_with_mrt_distances.csv` | Full dataset enriched with lat, lon, and nearest MRT columns |
| `../geocode_cache.json` | Cache of geocoded addresses (auto-created) |
| `../mrt_cache.json` | Cache of nearest MRT results (auto-created) |
| `../failed_geocodes.csv` | Addresses that could not be geocoded |

In [3]:
import pandas as pd
import json
import os
import time
import math
import requests
from tqdm.auto import tqdm
from dotenv import load_dotenv

print("Libraries imported successfully!")

Libraries imported successfully!


C:\Users\joelc\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Phase 1 — Extract Unique Addresses

Load the merged HDB dataset and extract all unique block + street_name combinations. These are the addresses we need to geocode — working with unique addresses avoids redundant API calls for the same location.

In [4]:
INPUT_CSV = '../merged_data/merged_hdb_resale_with_macro.csv'

print(f"Loading dataset from {INPUT_CSV}...")
df = pd.read_csv(INPUT_CSV)
print(f"Dataset shape: {df.shape}")

unique_addresses = df[['block', 'street_name']].drop_duplicates().reset_index(drop=True)
print(f"\nUnique addresses found: {len(unique_addresses):,}")
print("\nSample addresses:")
print(unique_addresses.head(10))

Loading dataset from ../merged_data/merged_hdb_resale_with_macro.csv...
Dataset shape: (197432, 19)

Unique addresses found: 9,660

Sample addresses:
  block        street_name
0   314   ANG MO KIO AVE 3
1   156   ANG MO KIO AVE 4
2   463  ANG MO KIO AVE 10
3   503   ANG MO KIO AVE 5
4   445  ANG MO KIO AVE 10
5   565   ANG MO KIO AVE 3
6   178   ANG MO KIO AVE 4
7   182   ANG MO KIO AVE 5
8   214   ANG MO KIO AVE 3
9   504   ANG MO KIO AVE 8


## Phase 2 — OneMap Authentication

Load credentials from `.env` and authenticate with the OneMap API. A shared `requests.Session` is created here and reused across all API calls — this enables connection pooling, which avoids the TCP handshake overhead on every request and meaningfully speeds up large loops.

In [8]:
load_dotenv('../.env')

ONEMAP_EMAIL = os.getenv('ONEMAP_EMAIL')
ONEMAP_PASSWORD = os.getenv('ONEMAP_PASSWORD')

if not ONEMAP_EMAIL or not ONEMAP_PASSWORD:
    raise SystemExit(
        "Credentials missing. Open ../.env and fill in:\n"
        "  ONEMAP_EMAIL=your_email\n"
        "  ONEMAP_PASSWORD=your_password"
    )

print("Credentials loaded successfully.")

Credentials loaded successfully.


In [9]:
# Shared session for connection pooling across all API calls
session = requests.Session()

_token = {"access_token": None, "expiry": 0}

def get_token():
    """Return a valid OneMap access token, refreshing if expired."""
    if _token["access_token"] and time.time() < int(_token["expiry"]) - 60:
        return _token["access_token"]

    resp = session.post(
        "https://www.onemap.gov.sg/api/auth/post/getToken",
        json={"email": ONEMAP_EMAIL, "password": ONEMAP_PASSWORD},
        timeout=10
    )
    resp.raise_for_status()
    data = resp.json()
    _token["access_token"] = data["access_token"]
    _token["expiry"] = int(data["expiry_timestamp"])
    return _token["access_token"]

# Verify authentication works
token = get_token()
expiry_str = time.strftime('%Y-%m-%d %H:%M:%S', time.localtime(_token["expiry"]))
print(f"Authentication successful!")
print(f"Token expires at: {expiry_str}")

Authentication successful!
Token expires at: 2026-02-22 16:29:37


## Phase 3 — Geocode Unique Addresses

Query the OneMap search API to retrieve lat/lon coordinates for each unique address. Results are cached to `geocode_cache.json` so the notebook can be safely interrupted and resumed — already-geocoded addresses are skipped. The cache is saved every 50 calls as a checkpoint.

In [19]:
STREET_ABBREVS = {
    'JLN': 'JALAN',
    'LOR': 'LORONG',
    'BT':  'BUKIT',
    'KG':  'KAMPONG',   # official Singapore spelling is KAMPONG, not KAMPUNG
}

def normalize_street(name: str) -> str:
    """Expand common Malay abbreviations for OneMap search queries."""
    return ' '.join(STREET_ABBREVS.get(token, token) for token in name.split())

# Verify expansions
print("Abbreviation expansion check:")
tests = [
    'JLN RUMAH TINGGI',
    'LOR 1 TOA PAYOH',
    'BT BATOK ST 11',
    'KG ARANG RD',
    'JLN BT MERAH',       # double expansion
    'ANG MO KIO AVE 3',   # no change expected
]
for t in tests:
    print(f"  {t!r:35s} -> {normalize_street(t)!r}")

Abbreviation expansion check:
  'JLN RUMAH TINGGI'                  -> 'JALAN RUMAH TINGGI'
  'LOR 1 TOA PAYOH'                   -> 'LORONG 1 TOA PAYOH'
  'BT BATOK ST 11'                    -> 'BUKIT BATOK ST 11'
  'KG ARANG RD'                       -> 'KAMPONG ARANG RD'
  'JLN BT MERAH'                      -> 'JALAN BUKIT MERAH'
  'ANG MO KIO AVE 3'                  -> 'ANG MO KIO AVE 3'


In [20]:
import random

# Singapore bounding box for accuracy validation
SG_LAT = (1.15, 1.50)
SG_LON = (103.60, 104.05)

def in_singapore(lat, lon):
    return SG_LAT[0] <= lat <= SG_LAT[1] and SG_LON[0] <= lon <= SG_LON[1]

def first_with(prefix):
    """Return the first real (block, street_name) from unique_addresses containing the prefix."""
    match = unique_addresses[unique_addresses['street_name'].str.startswith(prefix + ' ')]
    if match.empty:
        match = unique_addresses[unique_addresses['street_name'].str.contains(r'\b' + prefix + r'\b', regex=True)]
    row = match.iloc[0]
    return (str(row['block']), row['street_name'])

# Seed: one real address per Malay abbreviation type
seed_addresses = [
    first_with('JLN'),
    first_with('LOR'),
    first_with('BT'),
    first_with('KG'),
]
print("Seed addresses (real records from dataset):")
for b, s in seed_addresses:
    print(f"  {b} {s}  ->  query: {b} {normalize_street(s)}")

# Fill remaining 96 slots from unique_addresses (reproducible random sample)
random_sample = unique_addresses.sample(min(96, len(unique_addresses)), random_state=42)
sample_list = seed_addresses + [
    (str(row['block']), row['street_name']) for _, row in random_sample.iterrows()
]

print(f"\n=== SAMPLE GEOCODING TEST ({len(sample_list)} addresses) ===")
print(f"(Seed: {len(seed_addresses)} Malay abbreviation cases + {len(sample_list)-len(seed_addresses)} random)\n")

# In-memory cache — consumed by sample phases 4, 5, 6
sample_geocode_cache = {}

ok = failed = out_of_bounds = 0

for block, street_name in tqdm(sample_list, desc="Sample geocoding"):
    key   = f"{block} {street_name}"
    query = f"{block} {normalize_street(street_name)}"
    try:
        resp = session.get(
            GEOCODE_URL,
            params={'searchVal': query, 'returnGeom': 'Y', 'getAddrDetails': 'Y', 'pageNum': 1},
            headers={'Authorization': f'Bearer {get_token()}'},
            timeout=10
        )
        resp.raise_for_status()
        results = resp.json().get('results', [])
        if results:
            r = results[0]
            lat, lon = float(r['LATITUDE']), float(r['LONGITUDE'])
            returned_address = r.get('ADDRESS', '')
            flag = '' if in_singapore(lat, lon) else '  *** OUT OF BOUNDS ***'
            print(f"  OK   {key:45s} ({lat:.5f}, {lon:.5f}){flag}")
            print(f"         Returned: {returned_address}")
            sample_geocode_cache[key] = {'lat': lat, 'lon': lon,
                                         'postal': r.get('POSTAL', ''),
                                         'address': returned_address}
            ok += 1
            if flag:
                out_of_bounds += 1
        else:
            print(f"  FAIL {key:45s} (no results)")
            sample_geocode_cache[key] = None
            failed += 1
    except requests.RequestException as e:
        print(f"  ERR  {key:45s} ({e})")
        sample_geocode_cache[key] = None
        failed += 1
    time.sleep(0.3)

print(f"\nSample complete: {ok} succeeded, {failed} failed, {out_of_bounds} out-of-bounds.")
print(f"sample_geocode_cache has {len(sample_geocode_cache)} entries — ready for sample Phases 4, 5, 6.")
print("Review the returned addresses above, then run the full geocoding cell below.")

Seed addresses (real records from dataset):
  657 JLN TENAGA  ->  query: 657 JALAN TENAGA
  137 LOR AH SOO  ->  query: 137 LORONG AH SOO
  242 BT BATOK EAST AVE 5  ->  query: 242 BUKIT BATOK EAST AVE 5
  1 KG KAYU RD  ->  query: 1 KAMPONG KAYU RD

=== SAMPLE GEOCODING TEST (100 addresses) ===
(Seed: 4 Malay abbreviation cases + 96 random)



Sample geocoding:   0%|          | 0/100 [00:00<?, ?it/s]

  OK   657 JLN TENAGA                                (1.33383, 103.90709)
         Returned: 657 JALAN TENAGA EUNOS DAMAI VILLE SINGAPORE 410657


Sample geocoding:   1%|          | 1/100 [00:01<02:09,  1.31s/it]

  OK   137 LOR AH SOO                                (1.34905, 103.88664)
         Returned: 137 LORONG AH SOO SINGAPORE 530137


...

  OK   547 HOUGANG ST 51                             (1.37936, 103.89204)
         Returned: 547 HOUGANG STREET 51 SINGAPORE 530547


Sample geocoding:  99%|█████████▉| 99/100 [00:37<00:00,  2.76it/s]

  OK   297 CHOA CHU KANG AVE 2                       (1.37708, 103.74191)
         Returned: 297 CHOA CHU KANG AVENUE 2 SINGAPORE 680297


Sample geocoding: 100%|██████████| 100/100 [00:37<00:00,  2.65it/s]


Sample complete: 100 succeeded, 0 failed, 0 out-of-bounds.
sample_geocode_cache has 100 entries — ready for sample Phases 4, 5, 6.
Review the returned addresses above, then run the full geocoding cell below.


### [ FULL DATASET ] — Geocode all unique addresses
Run this after the sample above looks correct. Saves results to `geocode_cache.json`.

In [16]:
GEOCODE_CACHE_FILE = '../geocode_cache.json'
GEOCODE_URL = 'https://www.onemap.gov.sg/api/common/elastic/search'

# Load existing cache or start fresh
if os.path.exists(GEOCODE_CACHE_FILE):
    with open(GEOCODE_CACHE_FILE, 'r') as f:
        geocode_cache = json.load(f)
    print(f"Loaded existing geocode cache: {len(geocode_cache):,} entries")
else:
    geocode_cache = {}
    print("No existing cache found — starting fresh.")

# Only process addresses not already in cache
to_geocode = [
    (str(row['block']), row['street_name'])
    for _, row in unique_addresses.iterrows()
    if f"{row['block']} {row['street_name']}" not in geocode_cache
]
print(f"Addresses to geocode: {len(to_geocode):,} (skipping {len(geocode_cache):,} cached)")

call_count = 0
for block, street_name in tqdm(to_geocode, desc="Geocoding"):
    key   = f"{block} {street_name}"
    query = f"{block} {normalize_street(street_name)}"   # expanded query, original key
    try:
        resp = session.get(
            GEOCODE_URL,
            params={
                'searchVal': query,
                'returnGeom': 'Y',
                'getAddrDetails': 'Y',
                'pageNum': 1
            },
            headers={'Authorization': f'Bearer {get_token()}'},
            timeout=10
        )
        resp.raise_for_status()
        results = resp.json().get('results', [])
        if results:
            r = results[0]
            geocode_cache[key] = {
                'lat': float(r['LATITUDE']),
                'lon': float(r['LONGITUDE']),
                'postal': r.get('POSTAL', ''),
                'address': r.get('ADDRESS', '')
            }
        else:
            geocode_cache[key] = None
    except requests.RequestException as e:
        print(f"  [ERROR] {key}: {e}")
        geocode_cache[key] = None

    call_count += 1
    time.sleep(0.3)

    if call_count % 50 == 0:
        with open(GEOCODE_CACHE_FILE, 'w') as f:
            json.dump(geocode_cache, f)

# Final save
with open(GEOCODE_CACHE_FILE, 'w') as f:
    json.dump(geocode_cache, f)

success = sum(1 for v in geocode_cache.values() if v is not None)
failed  = sum(1 for v in geocode_cache.values() if v is None)
print(f"\nGeocoding complete.")
print(f"  Successfully geocoded: {success:,}")
print(f"  Failed / no results:   {failed:,}")

Loaded existing geocode cache: 500 entries
Addresses to geocode: 9,160 (skipping 500 cached)


Geocoding:  83%|████████▎ | 7622/9160 [1:05:19<45:22:10, 106.20s/it]

  [ERROR] 215 LOR 8 TOA PAYOH: ('Connection aborted.', ConnectionResetError(10054, 'An existing connection was forcibly closed by the remote host', None, 10054, None))


Geocoding:  92%|█████████▏| 8389/9160 [1:20:22<33:12:55, 155.09s/it]

  [ERROR] 116A JLN TENTERAM: ('Connection aborted.', ConnectionResetError(10054, 'An existing connection was forcibly closed by the remote host', None, 10054, None))


Geocoding: 100%|██████████| 9160/9160 [1:25:34<00:00,  1.78it/s]    



Geocoding complete.
  Successfully geocoded: 9,657
  Failed / no results:   3


## Phase 4 — Fetch Nearest MRT Station

For each successfully geocoded address, query the OneMap nearby services API to find the nearest MRT station. LRT stations are filtered out using a prefix check on the station ID. If no MRT is found within 2 km, the search radius is automatically expanded to 5 km. Results are cached to `mrt_cache.json` for resume safety.

### [ SAMPLE TEST ] — MRT lookup on 100 sample addresses
Run this first. Uses `sample_geocode_cache` from the Phase 3 sample cell. Results are kept in memory only (`sample_mrt_cache`).

In [31]:
# MRT line prefixes — used to exclude LRT stations (e.g. BP, PTC, PE, PW prefixes)
MRT_LINE_PREFIXES = ("NS", "EW", "CC", "DT", "NE", "TE", "CG", "CE", "JS", "JW")

# MRT station opening dates — Thomson–East Coast Line (TE) only.
# All other lines (NS, EW, NE, CC, DT, CG, CE) were fully open before April 2018.
# Station IDs verified against the OneMap getNearestMrtStops API.
# Source: LTA MRT network opening history.
MRT_OPENING_DATES = {
    # TE Stage 1 — 31 Jan 2020: Woodlands North, Woodlands, Woodlands South
    'TE1': '2020-01-31', 'TE2': '2020-01-31', 'TE3': '2020-01-31',
    # TE Stage 2 — 28 Aug 2021: Springleaf, Lentor, Mayflower, Bright Hill, Upper Thomson, Caldecott
    'TE4': '2021-08-28', 'TE5': '2021-08-28', 'TE6': '2021-08-28',
    'TE7': '2021-08-28', 'TE8': '2021-08-28', 'TE9': '2021-08-28',
    # TE Stage 3 — 13 Nov 2022: Mount Pleasant → Gardens by the Bay (TE10–TE22)
    'TE10': '2022-11-13', 'TE11': '2022-11-13', 'TE12': '2022-11-13',
    'TE13': '2022-11-13', 'TE14': '2022-11-13', 'TE15': '2022-11-13',
    'TE16': '2022-11-13', 'TE17': '2022-11-13', 'TE18': '2022-11-13',
    'TE19': '2022-11-13', 'TE20': '2022-11-13', 'TE21': '2022-11-13',
    'TE22': '2022-11-13',
    # TE Stage 4 — 23 Jun 2024: Tanjong Rhu → Bayshore (TE23–TE29)
    'TE23': '2024-06-23', 'TE24': '2024-06-23', 'TE25': '2024-06-23',
    'TE26': '2024-06-23', 'TE27': '2024-06-23', 'TE28': '2024-06-23',
    'TE29': '2024-06-23',
    # TE Stage 5 — 19 Nov 2024: Bedok South, Sungei Bedok
    'TE30': '2024-11-19', 'TE31': '2024-11-19',
}

def is_mrt(station_id: str) -> bool:
    """Return True only if station_id belongs to an MRT line (not LRT)."""
    return station_id.upper().startswith(MRT_LINE_PREFIXES)

def haversine_distance(lat1: float, lon1: float, lat2: float, lon2: float) -> float:
    """Return straight-line distance in kilometres between two lat/lon points."""
    R = 6371.0
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlambda = math.radians(lon2 - lon1)
    a = math.sin(dphi / 2) ** 2 + math.cos(phi1) * math.cos(phi2) * math.sin(dlambda / 2) ** 2
    return R * 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))

def get_nearest_mrt(flat_lat: float, flat_lon: float, token: str, radius: int = 2000, top_n: int = 5):
    """Return list of up to top_n nearest MRT stations sorted by distance, or [] if none found.

    top_n=5 ensures that even when several future TE stations cluster near an address,
    there is still room in the list for an older fallback station (e.g. EW, NS, DT).
    Phase 6 uses this ranked list to pick the nearest station open at transaction time.
    Each item: {'name': str, 'id': str, 'dist_m': float}
    """
    for r in [radius, 5000]:
        try:
            resp = session.get(
                'https://www.onemap.gov.sg/api/public/nearbysvc/getNearestMrtStops',
                params={'latitude': flat_lat, 'longitude': flat_lon, 'radius_in_meters': r},
                headers={'Authorization': token},
                timeout=10
            )
            resp.raise_for_status()
            stations = resp.json()
            if not isinstance(stations, list):
                stations = []
            mrt_stations = [s for s in stations if is_mrt(s.get('id', ''))]
            if mrt_stations:
                mrt_stations_sorted = sorted(
                    mrt_stations,
                    key=lambda s: haversine_distance(flat_lat, flat_lon, float(s['lat']), float(s['lon']))
                )[:top_n]
                return [
                    {
                        'name':   s['name'],
                        'id':     s['id'],
                        'dist_m': round(haversine_distance(flat_lat, flat_lon,
                                                           float(s['lat']), float(s['lon'])) * 1000, 1)
                    }
                    for s in mrt_stations_sorted
                ]
        except requests.RequestException:
            return []
        if r == radius:
            continue  # retry with 5000m
        break
    return []

print("Helper functions defined: is_mrt(), haversine_distance(), get_nearest_mrt()")
print(f"MRT_OPENING_DATES loaded: {len(MRT_OPENING_DATES)} TE stations tracked (Stages 1–5)")

Helper functions defined: is_mrt(), haversine_distance(), get_nearest_mrt()
MRT_OPENING_DATES loaded: 31 TE stations tracked (Stages 1–5)


In [32]:
sample_mrt_cache = {}

sample_to_lookup = [(k, v) for k, v in sample_geocode_cache.items() if v is not None]
print(f"Looking up MRT for {len(sample_to_lookup)} successfully geocoded sample addresses...\n")

for key, geo in tqdm(sample_to_lookup, desc="Sample MRT lookup"):
    try:
        result = get_nearest_mrt(geo['lat'], geo['lon'], get_token())
        sample_mrt_cache[key] = result  # list of up to 3 stations, or []
    except Exception as e:
        print(f"  [ERROR] {key}: {e}")
        sample_mrt_cache[key] = []
    time.sleep(0.3)

with_mrt = sum(1 for v in sample_mrt_cache.values() if v)
print(f"\nSample MRT lookup complete: {with_mrt} found, {len(sample_mrt_cache)-with_mrt} not found.")
print("\nPreview (first 5 results — showing nearest station):")
for key, val in list(sample_mrt_cache.items())[:5]:
    if val:
        nearest = val[0]
        print(f"  {key[:45]:45s} -> {nearest['id']:6} {nearest['name']:<30} {nearest['dist_m']:>7.1f}m")
    else:
        print(f"  {key[:45]:45s} -> None")

Looking up MRT for 100 successfully geocoded sample addresses...



Sample MRT lookup: 100%|██████████| 100/100 [00:41<00:00,  2.42it/s]


Sample MRT lookup complete: 100 found, 0 not found.

Preview (first 5 results — showing nearest station):
  657 JLN TENAGA                                -> DT28   KAKI BUKIT MRT STATION           187.3m
  137 LOR AH SOO                                -> CC12   BARTLEY MRT STATION             1033.6m
  242 BT BATOK EAST AVE 5                       -> NS2    BUKIT BATOK MRT STATION          773.9m
  1 KG KAYU RD                                  -> CC7    MOUNTBATTEN MRT STATION          328.0m
  616 WOODLANDS AVE 4                           -> TE3    WOODLANDS SOUTH MRT STATION      822.3m


### [ FULL DATASET ] — MRT lookup on all geocoded addresses
Run after the sample above looks correct. Saves results to `mrt_cache.json`.

In [35]:
MRT_CACHE_FILE = '../mrt_cache.json'

# ⚠️  If mrt_cache.json was created by an older version of this notebook (single-dict format),
#     delete it before running so the cache is rebuilt in the new list format.

# Load existing cache or start fresh
if os.path.exists(MRT_CACHE_FILE):
    with open(MRT_CACHE_FILE, 'r') as f:
        mrt_cache = json.load(f)
    print(f"Loaded existing MRT cache: {len(mrt_cache):,} entries")
else:
    mrt_cache = {}
    print("No existing MRT cache found — starting fresh.")

# Only process successfully geocoded addresses not already in mrt_cache
to_lookup = [
    (key, val)
    for key, val in geocode_cache.items()
    if val is not None and key not in mrt_cache
]
print(f"Addresses to look up: {len(to_lookup):,} (skipping {len(mrt_cache):,} cached)")

call_count = 0
for key, geo in tqdm(to_lookup, desc="MRT lookup"):
    try:
        result = get_nearest_mrt(geo['lat'], geo['lon'], get_token())
        mrt_cache[key] = result  # list of up to 3 stations, or []
    except Exception as e:
        print(f"  [ERROR] {key}: {e}")
        mrt_cache[key] = []

    call_count += 1
    time.sleep(0.3)

    if call_count % 50 == 0:
        with open(MRT_CACHE_FILE, 'w') as f:
            json.dump(mrt_cache, f)

# Final save
with open(MRT_CACHE_FILE, 'w') as f:
    json.dump(mrt_cache, f)

with_mrt = sum(1 for v in mrt_cache.values() if v)
print(f"\nMRT lookup complete.")
print(f"  Addresses with MRT result: {with_mrt:,}")
print(f"  Addresses with no result:  {len(mrt_cache) - with_mrt:,}")

No existing MRT cache found — starting fresh.
Addresses to look up: 9,657 (skipping 0 cached)


MRT lookup: 100%|██████████| 9657/9657 [1:40:07<00:00,  1.61it/s]     



MRT lookup complete.
  Addresses with MRT result: 9,653
  Addresses with no result:  4


## Phase 5 — Flag Failed Geocodes

Collect all addresses where geocoding returned no result and save them to `failed_geocodes.csv` for manual review. These rows will have null lat/lon and MRT values in the final dataset.

### [ SAMPLE TEST ] — Failed geocodes from sample
Prints failed addresses from `sample_geocode_cache`. No file is saved.

In [33]:
sample_failed = [k for k, v in sample_geocode_cache.items() if v is None]

print(f"Failed geocodes in sample: {len(sample_failed)}")
if sample_failed:
    print("\nFailed addresses:")
    for key in sample_failed:
        parts = key.split(' ', 1)
        print(f"  Block {parts[0]:6s}  {parts[1] if len(parts) > 1 else ''}")
else:
    print("No failures — all sample addresses geocoded successfully.")

Failed geocodes in sample: 0
No failures — all sample addresses geocoded successfully.


### [ FULL DATASET ] — Failed geocodes from full run
Saves all failed addresses to `failed_geocodes.csv`.

In [36]:
FAILED_FILE = '../failed_geocodes.csv'

failed_keys = [key for key, val in geocode_cache.items() if val is None]

failed_rows = []
for key in failed_keys:
    parts = key.split(' ', 1)
    block = parts[0]
    street = parts[1] if len(parts) > 1 else ''
    failed_rows.append({'block': block, 'street_name': street})

failed_df = pd.DataFrame(failed_rows)
failed_df.to_csv(FAILED_FILE, index=False)

print(f"Failed geocodes: {len(failed_df):,}")
print(f"Saved to: {FAILED_FILE}")
if not failed_df.empty:
    print("\nSample failed addresses:")
    print(failed_df.head(10))

Failed geocodes: 3
Saved to: ../failed_geocodes.csv

Sample failed addresses:
  block       street_name
0    36  JLN RUMAH TINGGI
1   215   LOR 8 TOA PAYOH
2  116A      JLN TENTERAM


## Phase 6 — Merge and Save

Map the geocoded coordinates and MRT results back onto every row of the full dataset using block + street_name as the join key. Save the enriched dataset to `hdb_with_mrt_distances.csv`.

### [ SAMPLE TEST ] — Merge and preview sample results
Filters the dataset to the 100 sample addresses, merges results, and saves to `hdb_sample_with_mrt_distances.csv`.

In [34]:
import re

SAMPLE_OUTPUT_FILE = '../merged_data/hdb_sample_with_mrt_distances.csv'

def pick_open_mrt(ranked_stations, transaction_month):
    """From a ranked list, return the nearest station that was open at transaction_month, or None."""
    for station in ranked_stations:
        sid = station['id']
        open_str = MRT_OPENING_DATES.get(sid)
        if open_str is None or transaction_month >= pd.Timestamp(open_str):
            return station
    return None

df_sample = df.copy()
df_sample['_key'] = df_sample['block'].astype(str) + ' ' + df_sample['street_name']
df_sample = df_sample[df_sample['_key'].isin(sample_geocode_cache.keys())].copy()
print(f"Rows matching sample addresses: {len(df_sample):,}")

df_sample['lat'] = df_sample['_key'].map(
    lambda k: sample_geocode_cache[k]['lat'] if sample_geocode_cache.get(k) else None)
df_sample['lon'] = df_sample['_key'].map(
    lambda k: sample_geocode_cache[k]['lon'] if sample_geocode_cache.get(k) else None)

print("Picking nearest open MRT per transaction...")
df_sample['month'] = pd.to_datetime(df_sample['month'])
df_sample['_chosen'] = df_sample.apply(
    lambda row: pick_open_mrt(sample_mrt_cache.get(row['_key'], []), row['month']),
    axis=1
)
df_sample['nearest_mrt_line']   = df_sample['_chosen'].apply(
    lambda s: re.match(r'^([A-Za-z]+)', s['id']).group(1) if s else None)
df_sample['nearest_mrt_dist_m'] = df_sample['_chosen'].apply(
    lambda s: s['dist_m'] if s else None)
df_sample = df_sample.drop(columns=['_key', '_chosen'])

df_sample.to_csv(SAMPLE_OUTPUT_FILE, index=False)

has_mrt     = df_sample['nearest_mrt_dist_m'].notna().sum()
missing_mrt = df_sample['nearest_mrt_dist_m'].isna().sum()
print(f"\n{'='*60}")
print("SAMPLE SUMMARY")
print(f"{'='*60}")
print(f"Total rows in sample output:  {len(df_sample):,}")
print(f"Rows with MRT distance:       {has_mrt:,}")
print(f"Rows missing MRT distance:    {missing_mrt:,}")
if has_mrt > 0:
    print(f"\nNearest MRT distance (m):")
    print(f"  Min:  {df_sample['nearest_mrt_dist_m'].min():.1f}")
    print(f"  Mean: {df_sample['nearest_mrt_dist_m'].mean():.1f}")
    print(f"  Max:  {df_sample['nearest_mrt_dist_m'].max():.1f}")
    print(f"\nMRT line distribution:")
    print(df_sample['nearest_mrt_line'].value_counts().to_string())
print(f"\nSaved to: {SAMPLE_OUTPUT_FILE}")
print("\nPreview:")
df_sample[['block', 'street_name', 'month', 'lat', 'lon', 'nearest_mrt_line', 'nearest_mrt_dist_m']].head(10)

Rows matching sample addresses: 1,853
Picking nearest open MRT per transaction...

SAMPLE SUMMARY
Total rows in sample output:  1,853
Rows with MRT distance:       1,853
Rows missing MRT distance:    0

Nearest MRT distance (m):
  Min:  74.3
  Mean: 848.1
  Max:  2342.0

MRT line distribution:
nearest_mrt_line
NS    580
DT    407
NE    368
EW    304
CC    117
TE     77

Saved to: ../merged_data/hdb_sample_with_mrt_distances.csv

Preview:


,block,street_name,month,lat,lon,nearest_mrt_line,nearest_mrt_dist_m
46,256,ANG MO KIO AVE 4,2018-04-01,1.370615,103.835968,NS,1515.6
142,108,BEDOK NTH RD,2018-04-01,1.332740,103.936224,DT,650.5
180,657,JLN TENAGA,2018-04-01,1.333831,103.907090,DT,187.3
239,242,BT BATOK EAST AVE 5,2018-04-01,1.350960,103.756233,NS,773.9
251,242,BT BATOK EAST AVE 5,2018-04-01,1.350960,103.756233,NS,773.9
302,183,BT BATOK WEST AVE 8,2018-04-01,1.346326,103.743378,NS,748.0
370,5,DELTA AVE,2018-04-01,1.292231,103.827601,EW,672.9
408,20,JLN MEMBINA,2018-04-01,1.285625,103.825923,EW,128.4
568,228,CHOA CHU KANG CTRL,2018-04-01,1.380948,103.746496,NS,519.4
722,137,LOR AH SOO,2018-04-01,1.349053,103.886640,CC,1033.6


### [ FULL DATASET ] — Merge and save final enriched dataset
Run after all sample phases look correct. Saves to `hdb_with_mrt_distances.csv`.

In [37]:
import re

OUTPUT_FILE = '../merged_data/hdb_with_mrt_distances.csv'

def pick_open_mrt(ranked_stations, transaction_month):
    """From a ranked list, return the nearest station that was open at transaction_month, or None."""
    for station in ranked_stations:
        sid = station['id']
        open_str = MRT_OPENING_DATES.get(sid)
        if open_str is None or transaction_month >= pd.Timestamp(open_str):
            return station
    return None

print("Mapping geocode results onto full dataset...")
df['_key'] = df['block'].astype(str) + ' ' + df['street_name']

df['lat'] = df['_key'].map(lambda k: geocode_cache[k]['lat'] if geocode_cache.get(k) else None)
df['lon'] = df['_key'].map(lambda k: geocode_cache[k]['lon'] if geocode_cache.get(k) else None)

print("Picking nearest open MRT per transaction...")
df['month'] = pd.to_datetime(df['month'])
df['_chosen'] = df.apply(
    lambda row: pick_open_mrt(mrt_cache.get(row['_key'], []), row['month']),
    axis=1
)
df['nearest_mrt_line']   = df['_chosen'].apply(
    lambda s: re.match(r'^([A-Za-z]+)', s['id']).group(1) if s else None)
df['nearest_mrt_dist_m'] = df['_chosen'].apply(
    lambda s: s['dist_m'] if s else None)
df = df.drop(columns=['_key', '_chosen'])

print(f"Saving to {OUTPUT_FILE}...")
df.to_csv(OUTPUT_FILE, index=False)

has_mrt     = df['nearest_mrt_dist_m'].notna().sum()
missing_mrt = df['nearest_mrt_dist_m'].isna().sum()
print(f"\n{'='*60}")
print("SUMMARY")
print(f"{'='*60}")
print(f"Total rows in output:         {len(df):,}")
print(f"Rows with MRT distance:       {has_mrt:,}")
print(f"Rows missing MRT distance:    {missing_mrt:,}")
if has_mrt > 0:
    print(f"\nNearest MRT distance (m):")
    print(f"  Min:  {df['nearest_mrt_dist_m'].min():.1f}")
    print(f"  Mean: {df['nearest_mrt_dist_m'].mean():.1f}")
    print(f"  Max:  {df['nearest_mrt_dist_m'].max():.1f}")
    print(f"\nMRT line distribution:")
    print(df['nearest_mrt_line'].value_counts().to_string())
print(f"\nNew columns added: lat, lon, nearest_mrt_line, nearest_mrt_dist_m")
print(f"\nDone! Output saved to {OUTPUT_FILE}")

Mapping geocode results onto full dataset...
Picking nearest open MRT per transaction...
Saving to ../merged_data/hdb_with_mrt_distances.csv...

SUMMARY
Total rows in output:         197,432
Rows with MRT distance:       197,113
Rows missing MRT distance:    319

Nearest MRT distance (m):
  Min:  39.7
  Mean: 782.7
  Max:  3531.8

MRT line distribution:
nearest_mrt_line
NS    59850
NE    48314
EW    46678
DT    28378
CC     8371
TE     5483
CG       39

New columns added: lat, lon, nearest_mrt_line, nearest_mrt_dist_m

Done! Output saved to ../merged_data/hdb_with_mrt_distances.csv
